# 01 - Data Exploration

Exploratory summary statistics for raw ADNI and PPMI data.  
This notebook loads raw data, shows participant flow, trajectory distributions,
visit spacing histograms, and protein detectability.  
**This notebook does not run any analyses** -- it imports from `src/` and reads saved results.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.adni_loader import load_adnimerge, load_somascan_adni, assign_conversion_labels
from src.preprocessing.ppmi_loader import load_ppmi_clinical, load_somascan_ppmi
from src.preprocessing.somascan_qc import identify_seq_columns

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Config loaded. Random seed:', config['random_seed'])

## Participant Flow Diagram

Show how many participants are enrolled, have proteomics, have adequate visits, and end up in the analysis.

In [ ]:
# Load ADNI clinical data and SomaScan proteomics
adnimerge = load_adnimerge(config)
somascan = load_somascan_adni(config)
seq_cols = identify_seq_columns(somascan)

n_enrolled = adnimerge['RID'].nunique()
n_with_proteomics = somascan['RID'].nunique()

# Participants with minimum visits for longitudinal analysis
min_visits = config['adni']['min_visits_longitudinal']
visit_counts = somascan.groupby('RID')['VISCODE'].nunique()
n_adequate_visits = (visit_counts >= min_visits).sum()

print(f'ADNI participant flow:')
print(f'  Enrolled in ADNIMERGE:        {n_enrolled}')
print(f'  With SomaScan proteomics:     {n_with_proteomics}')
print(f'  With >= {min_visits} proteomic visits:  {n_adequate_visits}')
print(f'  Number of proteins measured:  {len(seq_cols)}')

## Trajectory Group Distribution

Distribution of conversion labels: MCI converters, stable MCI, CN amyloid-negative, CN amyloid-positive, etc.

In [ ]:
labeled = assign_conversion_labels(adnimerge, config)

trajectory_counts = labeled.groupby('TRAJECTORY')['RID'].nunique().sort_values(ascending=False)
print('Trajectory group counts (unique participants):')
print(trajectory_counts)

fig, ax = plt.subplots(figsize=(8, 4))
trajectory_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Participants')
ax.set_title('ADNI Trajectory Group Distribution')
plt.tight_layout()
plt.show()

## Visit Spacing Distribution

Histogram of inter-visit intervals (months) for participants with proteomics data.  
Valid intervals per config: between `visit_gap_min_months` and `visit_gap_max_months`.

In [ ]:
# Compute inter-visit intervals for participants with proteomics
merged = adnimerge.merge(somascan[['RID', 'VISCODE']], on=['RID', 'VISCODE'], how='inner')
merged = merged.sort_values(['RID', 'EXAMDATE'])

intervals = []
for rid, grp in merged.groupby('RID'):
    dates = grp['EXAMDATE'].dropna().sort_values()
    if len(dates) > 1:
        diffs = dates.diff().dt.days.dropna() / 30.44  # approximate months
        intervals.extend(diffs.tolist())

intervals = np.array(intervals)
gap_min = config['adni']['visit_gap_min_months']
gap_max = config['adni']['visit_gap_max_months']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(intervals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(gap_min, color='red', linestyle='--', label=f'Min valid gap ({gap_min}m)')
ax.axvline(gap_max, color='red', linestyle='--', label=f'Max valid gap ({gap_max}m)')
ax.set_xlabel('Inter-visit interval (months)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Inter-Visit Intervals (ADNI Proteomics)')
ax.legend()
plt.tight_layout()
plt.show()

valid_frac = np.mean((intervals >= gap_min) & (intervals <= gap_max))
print(f'{valid_frac:.1%} of intervals fall within the valid range ({gap_min}-{gap_max} months)')

## Protein Detectability

Fraction of samples where each protein is detected above LLOD.  
Proteins below the `min_detectability` threshold will be removed during QC.

In [ ]:
# Compute detectability per protein (fraction of non-NaN values)
detectability = somascan[seq_cols].notna().mean().sort_values()
min_detect = config['proteomics']['min_detectability']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(detectability, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(min_detect, color='red', linestyle='--', label=f'Threshold ({min_detect})')
ax.set_xlabel('Detectability (fraction of samples)')
ax.set_ylabel('Number of Proteins')
ax.set_title('Protein Detectability Distribution')
ax.legend()
plt.tight_layout()
plt.show()

n_pass = (detectability >= min_detect).sum()
n_fail = (detectability < min_detect).sum()
print(f'Proteins passing detectability filter: {n_pass}')
print(f'Proteins removed: {n_fail}')